# MLP comparison: four transforms × BLOSUM100

Train simple MLP classifiers on Fourier, circular Z, finite discrete Mellin, and
wavelet coefficients of the **same corrected BLOSUM100 embedding**. Preserve the
supplied ABP_GN train/test split and select configurations using training validation
only. Run all cells once, then edit the controls and rerun downstream cells.

This is Gram-negative antibacterial peptide classification. The test set has been
used in earlier experiments, so this is exploratory rather than independent
confirmation. No sequence-family separation is claimed. Raw data are not bundled.
Requires this development checkout; the published 0.1.0a1 lacks these transforms.
Install `python -m pip install -e '.[ml,wavelets,validation]'` plus `nbformat`,
`nbclient`, and `ipykernel` if executing programmatically.

In [ ]:
from pathlib import Path
import sys, json, hashlib, warnings, platform
from datetime import datetime, timezone
from uuid import uuid4
from time import perf_counter
import importlib.metadata
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "protspect").is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
from threadpoolctl import threadpool_limits
from protspect import transform_sequence
from protspect.embeddings import b100
from examples.validate_amp import load_data, metrics
%matplotlib inline

## Controls

Defaults use all 256 coefficients per channel (5,120 features per peptide), not
the dyadic Fourier selector in the earlier audit. Haar + periodization has the
same coefficient count; other wavelet choices may change it. The same hidden
architecture is not parameter-matched if feature widths differ.

All four defaults use **amplitudes** for consistency; this loses wavelet signs.
Set the wavelet component to `real` to retain signs as a separate experiment.
Z radius 1 duplicates Fourier; 1.01 supplies modest positional downweighting.
Mellin uses the finite positive-position sum documented in `docs/TRANSFORMS.md`.
Scaling is learned only within each fitting partition. A fixed grid is chosen
in advance and rejects longer sequences; sequences are never truncated.

In [ ]:
# Edit paths if the CSVs are elsewhere. Prefer ignored research/data when present.
DATA_DIR = ROOT / "research" / "data"
if not (DATA_DIR / "ABP_GN_train.csv").exists():
    DATA_DIR = Path.home() / "Downloads" / "input_csv"
TRAIN_CSV = DATA_DIR / "ABP_GN_train.csv"
TEST_CSV = DATA_DIR / "ABP_GN_test.csv"
SPLIT_SEED = 7
VALIDATION_FRACTION = 0.20
MODEL_SEEDS = [7]  # Add seeds to assess fitting variability; keep the split fixed.
LENGTH = 256
CENTER = False
NORMALIZATION = "none"
TRANSFORMS = {
    "fourier": {"component": "amplitude", "options": {}},
    "z": {"component": "amplitude", "options": {"radius": 1.01}},
    "mellin": {"component": "amplitude", "options": {"mellin_c": 0.0}},
    "wavelet": {"component": "amplitude", "options": {
        "wavelet": "haar", "level": 4, "mode": "periodization"}},
}
# Same candidate list for every transform. One candidate keeps the baseline simple.
MLP_CANDIDATES = [{"hidden_layer_sizes": (64,), "alpha": 1e-3}]
# To tune, add e.g. {"hidden_layer_sizes": (128, 64), "alpha": 1e-3} above.
MLP_COMMON = dict(activation="relu", solver="adam", learning_rate_init=1e-3,
    batch_size=128, max_iter=300, early_stopping=True, validation_fraction=.15,
    n_iter_no_change=15, tol=1e-4)
THRESHOLD = 0.5  # metrics() uses this fixed, untuned threshold.
THREADS = 1
SAVE_RESULTS = True
OUTPUT_BASE = ROOT / "validation" / "amp_transform_mlp"

In [ ]:
train_sequences, y_train = load_data(TRAIN_CSV)
test_sequences, y_test = load_data(TEST_CSV)
if len(set(train_sequences)) != len(train_sequences) or len(set(test_sequences)) != len(test_sequences):
    raise ValueError("Duplicate sequences within a split; resolve before benchmarking")
if set(train_sequences) & set(test_sequences):
    raise ValueError("Train/test sequence overlap")
if max(map(len, train_sequences + test_sequences)) > LENGTH:
    raise ValueError("A peptide exceeds LENGTH; increase the common grid, never truncate")
if THRESHOLD != .5:
    raise ValueError("This notebook uses the fixed 0.5 threshold")
if not MLP_CANDIDATES or not MODEL_SEEDS or len(set(MODEL_SEEDS)) != len(MODEL_SEEDS):
    raise ValueError("Provide candidates and unique model seeds")
fit_indices, validation_indices = train_test_split(np.arange(len(y_train)),
    test_size=VALIDATION_FRACTION, random_state=SPLIT_SEED, stratify=y_train)
source_hashes = {k: hashlib.sha256(p.read_bytes()).hexdigest()
                 for k,p in {"train": TRAIN_CSV, "test": TEST_CSV}.items()}
display(pd.DataFrame([
    {"split": "train-fit", "n": len(fit_indices), "positive": int(y_train[fit_indices].sum())},
    {"split": "train-validation", "n": len(validation_indices), "positive": int(y_train[validation_indices].sum())},
    {"split": "test", "n": len(y_test), "positive": int(y_test.sum())}]))
reference = ROOT / "validation" / "abp_gn_fft1024_seed7"
if (reference / "report.json").exists():
    audit = json.loads((reference / "report.json").read_text())
    same_data = all(source_hashes[k] == audit["data"][k]["sha256"] for k in source_hashes)
    saved = np.load(reference / "split_indices.npz")
    print("Same input files as previous audit:", same_data)
    print("Same training/validation indices:", np.array_equal(saved["fit"], fit_indices)
          and np.array_equal(saved["validation"], validation_indices))
print("Embedding: corrected BLOSUM100; channels:", len(b100))

## Featurize, validate, refit, and evaluate

Feature extraction has no fitted statistics: the embedding and grid are fixed,
and normalization is per sequence. Only the scaler and MLP learn from data.
Each candidate is fit on `train-fit`, ranked by validation balanced accuracy,
then MCC and average precision. The winner is refit on the full supplied training
set before test scoring. MLP early stopping uses an internal training-only subset
and monitors accuracy; it does not use the outer validation or test labels.

Feature matrices are processed one transform at a time to limit memory. This
cell may take several minutes. Re-running recomputes features and refits all models.
Changing seeds repeats fitting, not dataset splitting. Test values are never used
in candidate selection. Do not select later settings based on test results.

In [ ]:
def featurize(sequences, method, config):
    """Fixed channel-major coefficients, with no dataset-level fitted statistics."""
    output = np.empty((len(sequences), 0), dtype=np.float32)
    names = None
    for i, sequence in enumerate(sequences):
        result = transform_sequence(sequence, b100, method=method, length=LENGTH,
            center=CENTER, normalization=NORMALIZATION, **config["options"])
        row, current_names = result.features(config["component"])
        if names is None:
            names = current_names
            output = np.empty((len(sequences), len(row)), dtype=np.float32)
        if not np.array_equal(names, current_names):
            raise ValueError("Inconsistent feature schema")
        output[i] = row
    if not np.isfinite(output).all():
        raise ValueError("Nonfinite features after float32 conversion")
    return output, names, {k:v for k,v in result.metadata.items() if k != "sequence_length"}


def make_mlp(candidate, seed):
    return make_pipeline(StandardScaler(), MLPClassifier(
        random_state=seed, **MLP_COMMON, **candidate))


def fit_with_warnings(model, x, y):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(x, y)
    return [str(w.message) for w in caught]


def parameter_count(model):
    mlp = model.steps[-1][1]
    return int(sum(v.size for v in mlp.coefs_ + mlp.intercepts_))

In [ ]:
rows, trials, predictions, learning_curves, feature_info = [], [], {}, {}, {}
benchmark_start = perf_counter()
with threadpool_limits(limits=THREADS):
    for method, config in TRANSFORMS.items():
        start = perf_counter()
        X_train, names, metadata = featurize(train_sequences, method, config)
        X_test, test_names, _ = featurize(test_sequences, method, config)
        assert np.array_equal(names, test_names)
        feature_seconds = perf_counter() - start
        feature_info[method] = {"count": len(names), "names": names.tolist(),
            "metadata": metadata, "component": config["component"], "seconds": feature_seconds}
        print(f"{method}: {X_train.shape[1]:,} features; extraction {feature_seconds:.1f}s", flush=True)
        for seed in MODEL_SEEDS:
            start = perf_counter()
            best = None
            for candidate_index, candidate in enumerate(MLP_CANDIDATES):
                model = make_mlp(candidate, seed)
                messages = fit_with_warnings(model, X_train[fit_indices], y_train[fit_indices])
                probabilities = model.predict_proba(X_train[validation_indices])[:, 1]
                score = metrics(y_train[validation_indices], probabilities)
                trials.append({"transform": method, "seed": seed, "candidate": candidate_index,
                    "settings": candidate, "validation": score, "warnings": messages})
                rank = tuple(score[k] for k in ("balanced_accuracy", "mcc", "pr_auc"))
                if best is None or rank > best[0]:
                    best = rank, candidate, score
            _, selected, validation_score = best
            final = make_mlp(selected, seed)
            messages = fit_with_warnings(final, X_train, y_train)
            probability = final.predict_proba(X_test)[:, 1]
            test_score = metrics(y_test, probability)
            predictions[(method, seed)] = probability
            learning_curves[(method, seed)] = list(final.steps[-1][1].loss_curve_)
            rows.append({"transform": method, "seed": seed, "features": X_train.shape[1],
                "parameters": parameter_count(final), "selected": json.dumps(selected),
                "validation_BA": validation_score["balanced_accuracy"],
                **{"test_"+k:v for k,v in test_score.items()},
                "fit_seconds": perf_counter()-start, "epochs": final.steps[-1][1].n_iter_,
                "refit_warnings": "; ".join(messages)})
            print(f"  seed {seed}: test BA={test_score['balanced_accuracy']:.3f}, "
                  f"MCC={test_score['mcc']:.3f}; epochs={rows[-1]['epochs']}", flush=True)
        del X_train, X_test
results = pd.DataFrame(rows)
print(f"Total: {(perf_counter()-benchmark_start)/60:.1f} min")

## Compare performance

AP below means **average precision**, not trapezoidal PR area. All thresholded
metrics use 0.5. Repeated-seed standard deviations describe fitting variability;
they do not measure uncertainty from new datasets or sequence families. With one
seed the standard deviation is undefined. Feature widths and MLP parameter counts
are shown so capacity differences remain visible.

In [ ]:
display(results[["transform", "seed", "features", "parameters", "validation_BA",
    "test_balanced_accuracy", "test_mcc", "test_roc_auc", "test_pr_auc",
    "test_accuracy", "test_f1", "epochs", "fit_seconds", "refit_warnings"]])
metric_columns = ["test_balanced_accuracy", "test_mcc", "test_roc_auc", "test_pr_auc"]
summary = results.groupby("transform", sort=False)[metric_columns].agg(["mean", "std"])
display(summary)
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
means = results.groupby("transform", sort=False)[metric_columns].mean()
means.rename(columns={"test_balanced_accuracy": "Balanced accuracy", "test_mcc": "MCC",
    "test_roc_auc": "ROC-AUC", "test_pr_auc": "Average precision"}).plot.bar(ax=ax)
ax.set_ylim(-.1, 1); ax.set_ylabel("Test score (mean over model seeds)")
ax.tick_params(axis="x", rotation=0); ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()
warning_trials = [trial for trial in trials if trial["warnings"]]
print("Validation fits with warnings:", len(warning_trials))
if warning_trials:
    display(pd.DataFrame(warning_trials))

In [ ]:
# Use the first configured seed for curves; no best-test-seed selection.
PLOT_SEED = MODEL_SEEDS[0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for method in TRANSFORMS:
    probability = predictions[(method, PLOT_SEED)]
    RocCurveDisplay.from_predictions(y_test, probability, name=method, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, probability, name=method, ax=axes[1])
    axes[2].plot(learning_curves[(method, PLOT_SEED)], label=method)
axes[0].plot([0, 1], [0, 1], "k--", alpha=.4)
axes[1].axhline(y_test.mean(), color="k", linestyle="--", alpha=.4)
axes[2].set(xlabel="Epoch", ylabel="Training loss", title=f"Full-training refit: seed {PLOT_SEED}")
axes[2].legend(); plt.show()

## Save the run

Save CSV metrics, all validation trials, test probabilities (row indices only),
exact train/validation indices, feature definitions, source hashes, and environment
versions in a new timestamped folder. No raw peptide sequences or trained models
are exported. Run the notebook in order after edits so saved controls match results.

In [ ]:
if SAVE_RESULTS:
    output = OUTPUT_BASE / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid4().hex[:8])
    output.mkdir(parents=True, exist_ok=False)
    results.to_csv(output / "metrics.csv", index=False)
    summary.to_csv(output / "summary.csv")
    prediction_table = pd.DataFrame({"test_row": np.arange(len(y_test)), "label": y_test})
    for (method, seed), probability in predictions.items():
        prediction_table[f"{method}_seed{seed}"] = probability
    prediction_table.to_csv(output / "test_predictions.csv", index=False)
    np.savez_compressed(output / "split_indices.npz", fit=fit_indices, validation=validation_indices)
    versions = {p: importlib.metadata.version(p) for p in
                ["numpy", "scipy", "pandas", "scikit-learn", "PyWavelets", "matplotlib"]}
    source_files = ["protspect/transforms.py", "protspect/embeddings.py",
                    "protspect/preprocessing.py", "examples/validate_amp.py", "research/AMP_transform_MLP.ipynb"]
    report = {"task": "ABP_GN Gram-negative antibacterial peptide classification",
        "data_sha256": source_hashes, "train_n": len(y_train), "test_n": len(y_test),
        "split_seed": SPLIT_SEED, "validation_fraction": VALIDATION_FRACTION,
        "model_seeds": MODEL_SEEDS, "length": LENGTH, "center": CENTER,
        "normalization": NORMALIZATION, "transforms": TRANSFORMS,
        "embedding": "corrected BLOSUM100", "candidates": MLP_CANDIDATES,
        "mlp_common": MLP_COMMON, "threshold": THRESHOLD, "threads": THREADS,
        "selection": "validation balanced accuracy, MCC, average precision; test not used",
        "test_status": "previously studied test set; exploratory",
        "feature_info": feature_info, "trials": trials, "versions": versions,
        "python": platform.python_version(),
        "source_sha256": {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest() for f in source_files}}
    (output / "report.json").write_text(json.dumps(report, indent=2)+"\n")
    print("Saved:", output)
else:
    print("Saving disabled")